# LeetCode 1290 — Convert Binary Number in a Linked List to Integer bằng Python

**Link đề:** https://leetcode.com/problems/convert-binary-number-in-a-linked-list-to-integer/
**Độ khó:** Easy · **Chủ đề:** Linked List, Math

```python
# Definition for singly-linked list.
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
class Solution:
    def getDecimalValue(self, head: ListNode | None) -> int:
        ...
```

## Mục tiêu học
1. Hiểu **danh sách liên kết đơn** (singly linked list): nút, con trỏ `next`, duyệt bằng `while node:`.
2. Hiểu **hệ nhị phân**: giá trị theo vị trí `2^k`, đổi nhị phân → thập phân.
3. Đi từ cách "gom vào chuỗi rồi `int(s, 2)`" → "gom vào list rồi cộng lũy thừa" → **Horner một lượt** `num = num * 2 + bit`, `O(1)` bộ nhớ.
4. Dùng phép **dịch bit** `<<` và **OR** `|` để viết `num = (num << 1) | bit`.
5. Trace, chứng minh invariant, test đối chiếu, lỗi hay gặp, biến thể và bài tập có đáp án.

## 1. Đề bài và ví dụ

Cho `head` là nút đầu của một danh sách liên kết đơn. Mỗi nút có giá trị `0` hoặc `1`. Danh sách biểu diễn **một số nhị phân**, với **bit có trọng số cao nhất (MSB) ở đầu danh sách**. Trả về giá trị **thập phân** của số đó.

| `head` | nhị phân | Output |
|---|---|---:|
| `[1,0,1]` | `101₂` | `5` |
| `[0]` | `0₂` | `0` |

**Constraints:** danh sách không rỗng; tối đa **30** nút; mỗi giá trị là `0` hoặc `1`.

Vì tối đa 30 bit, kết quả `< 2^30` — vừa số nguyên 32-bit.

## 2. Nền tảng: danh sách liên kết trong Python

### 2.1. Lớp `ListNode`

LeetCode đã định nghĩa sẵn `ListNode`, nhưng để notebook **chạy độc lập**, ta tự định nghĩa lại giống hệt. Mỗi nút giữ:
- `val`: giá trị (ở bài này là bit 0/1),
- `next`: tham chiếu tới nút kế tiếp, hoặc `None` nếu là nút cuối.

In [1]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

    def __repr__(self):                     # để print ra dễ đọc
        return f"ListNode({self.val})"

# Tạo tay danh sách 1 -> 0 -> 1
c = ListNode(1)
b_ = ListNode(0, c)
a = ListNode(1, b_)
print(a, a.next, a.next.next, a.next.next.next)   # ListNode(1) ListNode(0) ListNode(1) None

ListNode(1) ListNode(0) ListNode(1) None


`__init__` là **hàm khởi tạo** chạy khi viết `ListNode(...)`; `self` là chính đối tượng đang tạo. `__repr__` quy định chuỗi hiển thị khi `print` một đối tượng.

### 2.2. Hàm dựng và in danh sách

Tạo tay từng nút rất mỏi tay. Ta viết `build_list([1, 0, 1])` và `to_list(head)` để chuyển qua lại giữa Python list và linked list.

In [2]:
def build_list(values):
    dummy = ListNode()              # nút giả đứng trước đầu danh sách
    tail = dummy
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next


def to_list(head):
    out = []
    node = head
    while node:                     # node khác None
        out.append(node.val)
        node = node.next
    return out


def show(head):
    return " -> ".join(str(v) for v in to_list(head)) + " -> None"

head = build_list([1, 0, 1])
print(to_list(head))
print(show(head))

[1, 0, 1]
1 -> 0 -> 1 -> None


**Nút giả (dummy)** giúp không phải xử lý riêng trường hợp "danh sách đang rỗng" khi nối nút đầu tiên. `while node:` dừng khi `node` là `None` (vì `None` được coi là False).

### 2.3. Duyệt danh sách: khuôn mẫu cần thuộc

```python
node = head
while node:
    ... xử lý node.val ...
    node = node.next
```

Không có chỉ số `i` như list; muốn tới nút thứ k phải đi `k` bước. Ta **không biết trước độ dài** danh sách.

In [3]:
node = build_list([1, 1, 0, 1])
step = 0
while node:
    print(f"bước {step}: val = {node.val}")
    node = node.next
    step += 1

bước 0: val = 1
bước 1: val = 1
bước 2: val = 0
bước 3: val = 1


## 3. Nền tảng: hệ nhị phân

### 3.1. Giá trị theo vị trí

Số nhị phân `b_{k-1} … b_1 b_0` có giá trị `Σ b_i · 2^i` (bit phải nhất là `2^0`).

```text
1 0 1  =  1·2² + 0·2¹ + 1·2⁰  =  4 + 0 + 1  =  5
1 1 0 1 = 8 + 4 + 0 + 1 = 13
```

In [4]:
print(1 * 2**2 + 0 * 2**1 + 1 * 2**0)    # 5
print(int("101", 2), int("1101", 2))     # 5 13
print(bin(5), bin(13))                   # 0b101 0b1101

5
5 13
0b101 0b1101


### 3.2. Dịch bit `<<` và OR `|`

- `x << 1` = dời mọi bit sang trái 1 vị trí, thêm `0` ở cuối = **nhân 2**.
- `x | 1` = bật bit cuối (nếu đang là 0) = cộng 1 khi bit cuối đang là 0.
- Nên với `bit ∈ {0, 1}`: `(x << 1) | bit == x * 2 + bit`.

In [5]:
x = 0b101                  # 5
print(bin(x << 1), x << 1)          # 0b1010 10
print(bin((x << 1) | 1), (x << 1) | 1)   # 0b1011 11
for x in range(8):
    for bit in (0, 1):
        assert (x << 1) | bit == x * 2 + bit
print("(x << 1) | bit == 2*x + bit với mọi x < 8, bit ∈ {0,1}")

0b1010 10
0b1011 11
(x << 1) | bit == 2*x + bit với mọi x < 8, bit ∈ {0,1}


## 4. Cách 1 — Gom thành chuỗi rồi `int(s, 2)`

Đúng như hint thứ nhất của đề: duyệt danh sách, nối các bit thành chuỗi, rồi để Python đổi cơ số.

In [6]:
def decimal_via_string(head):
    bits = []
    node = head
    while node:
        bits.append(str(node.val))
        node = node.next
    return int("".join(bits), 2)

print(decimal_via_string(build_list([1, 0, 1])))     # 5
print(decimal_via_string(build_list([0])))           # 0

5
0


**Độ phức tạp:** gọi `n` = số nút. Thời gian `O(n)`, bộ nhớ `O(n)` (list + chuỗi).

## 5. Cách 2 — Gom vào list rồi cộng lũy thừa (tự đổi cơ số)

Vấn đề: bit đầu danh sách có trọng số **lớn nhất** `2^(n-1)`, mà ta **không biết `n`** cho tới khi đi hết danh sách. Cách đơn giản: gom vào list trước để biết độ dài.

In [7]:
def decimal_via_powers(head):
    bits = to_list(head)
    n = len(bits)
    total = 0
    for i, bit in enumerate(bits):
        total += bit * 2 ** (n - 1 - i)      # bit thứ i (từ trái) có trọng số 2^(n-1-i)
    return total

print(decimal_via_powers(build_list([1, 0, 1])))      # 5
print(decimal_via_powers(build_list([1, 1, 0, 1])))   # 13

5
13


Hoặc duyệt **ngược** list với `reversed`, trọng số tăng dần từ `2^0`:

In [8]:
def decimal_via_reversed(head):
    total, weight = 0, 1
    for bit in reversed(to_list(head)):
        total += bit * weight
        weight *= 2
    return total

print(decimal_via_reversed(build_list([1, 1, 0, 1])))   # 13

13


## 6. Nút thắt

Cả hai cách trên đều phải **lưu toàn bộ bit** (bộ nhớ `O(n)`) chỉ vì không biết trước bit đầu "đáng giá" bao nhiêu.

Câu hỏi: có cách nào xử lý bit **ngay khi đọc**, từ trái sang phải, mà không cần biết độ dài?

## 7. Insight — Horner: "đọc thêm một bit = nhân đôi rồi cộng bit"

Giống hệt khi đọc số thập phân từ trái: đọc `1`, `2`, `3` → `1` → `12 = 1·10 + 2` → `123 = 12·10 + 3`.

Với nhị phân, thay `10` bằng `2`:

```text
đọc 1: num = 0·2 + 1 = 1      (1₂)
đọc 0: num = 1·2 + 0 = 2      (10₂)
đọc 1: num = 2·2 + 1 = 5      (101₂)
```

Mỗi lần nhân 2, **mọi bit đã đọc tự động được nâng trọng số lên 1 bậc** — nên ta không cần biết trước độ dài.

In [9]:
num = 0
for ch in "123":
    num = num * 10 + int(ch)
    print("thập phân:", num)
num = 0
for bit in [1, 0, 1]:
    num = num * 2 + bit
    print("nhị phân :", num)

thập phân: 1
thập phân: 12
thập phân: 123
nhị phân : 1
nhị phân : 2
nhị phân : 5


## 8. Cách 3 — Horner một lượt (lời giải chính)

In [10]:
def decimal_horner(head):
    num = 0
    node = head
    while node:
        num = num * 2 + node.val
        node = node.next
    return num

print(decimal_horner(build_list([1, 0, 1])))          # 5
print(decimal_horner(build_list([0])))                # 0
print(decimal_horner(build_list([1, 1, 0, 1])))       # 13

5
0
13


## 9. Cách 4 — Phiên bản phép toán bit

Thay `* 2` bằng `<< 1` và `+` bằng `|` (đúng như hint thứ hai của đề). Kết quả giống hệt; phép bit thể hiện rõ ý "ghép thêm một bit vào cuối".

In [11]:
def decimal_bitwise(head):
    num = 0
    while head:
        num = (num << 1) | head.val
        head = head.next
    return num

print(decimal_bitwise(build_list([1, 0, 1])), decimal_bitwise(build_list([1, 1, 0, 1])))   # 5 13

5 13


Lưu ý: ở đây ta "tái sử dụng" biến `head` để duyệt. Trong hàm thì không sao (chỉ đổi biến cục bộ, danh sách gốc vẫn nguyên), nhưng nếu sau đó cần dùng lại đầu danh sách thì phải giữ bản sao `node = head`.

## 10. Cách 5 — Đệ quy (tham khảo)

Đệ quy xử lý từ **cuối** về: hàm trả về cặp `(giá_trị_phần_sau, số_bit_phần_sau)`, rồi cộng bit hiện tại với trọng số `2^số_bit`.

In [12]:
def _rec(node):
    if node is None:
        return 0, 0                      # (giá trị, số bit)
    val_rest, len_rest = _rec(node.next)
    return node.val * (1 << len_rest) + val_rest, len_rest + 1


def decimal_recursive(head):
    return _rec(head)[0]

print(decimal_recursive(build_list([1, 0, 1])), decimal_recursive(build_list([1, 1, 0, 1])))   # 5 13

5 13


Đệ quy dùng `O(n)` ngăn xếp lời gọi; với `n ≤ 30` ổn, nhưng Python giới hạn độ sâu đệ quy (~1000) nên không phù hợp cho danh sách rất dài.

## 11. Trace từng bước (Cách 3)

`head = [1, 1, 0, 1]`:

| bước | `node.val` | `num` trước | `num * 2 + val` | `num` sau (nhị phân) |
|---:|---:|---:|---|---|
| 1 | 1 | 0 | 0·2 + 1 | 1 (`1`) |
| 2 | 1 | 1 | 1·2 + 1 | 3 (`11`) |
| 3 | 0 | 3 | 3·2 + 0 | 6 (`110`) |
| 4 | 1 | 6 | 6·2 + 1 | 13 (`1101`) |

`node` thành `None` → dừng, trả `13`.

In [13]:
def trace_horner(head):
    num = 0
    node = head
    step = 1
    while node:
        new = num * 2 + node.val
        print(f"bước {step}: val={node.val} | num={num:>2} -> {num}*2+{node.val} = {new:>2} (nhị phân {bin(new)[2:]})")
        num = new
        node = node.next
        step += 1
    print("node = None -> trả", num)
    return num

trace_horner(build_list([1, 1, 0, 1]))

bước 1: val=1 | num= 0 -> 0*2+1 =  1 (nhị phân 1)
bước 2: val=1 | num= 1 -> 1*2+1 =  3 (nhị phân 11)
bước 3: val=0 | num= 3 -> 3*2+0 =  6 (nhị phân 110)
bước 4: val=1 | num= 6 -> 6*2+1 = 13 (nhị phân 1101)
node = None -> trả 13


13

## 12. Code nộp LeetCode

Trên LeetCode **không** cần định nghĩa lại `ListNode` (đã có sẵn). Trong notebook, class `ListNode` ở mục 2 đã tồn tại nên cell dưới chạy được.

In [14]:
class Solution:
    def getDecimalValue(self, head: "ListNode | None") -> int:
        num = 0
        node = head
        while node:
            num = num * 2 + node.val
            node = node.next
        return num

print(Solution().getDecimalValue(build_list([1, 0, 1])))   # 5
print(Solution().getDecimalValue(build_list([0])))         # 0

5
0


Ta viết kiểu `"ListNode | None"` trong dấu nháy (forward reference) để không phụ thuộc thứ tự định nghĩa. Trên LeetCode bạn cứ giữ nguyên chữ ký `head: Optional[ListNode]` hoặc `ListNode | None` như khung đề.

## 13. Vì sao đúng? (invariant)

Gọi các bit là `b_0, b_1, …, b_{n-1}` (từ đầu danh sách). **Invariant:** sau khi đọc `k` bit đầu, `num` = giá trị của số nhị phân `b_0 b_1 … b_{k-1}`.

- `k = 0`: `num = 0` = giá trị của số rỗng. Đúng.
- Bước `k → k+1`: số `b_0 … b_k` = (số `b_0 … b_{k-1}`) dời trái 1 bit rồi thêm `b_k` vào cuối = `num · 2 + b_k`. Đúng với cập nhật.
- Khi `node = None`, đã đọc đủ `n` bit → `num` là đáp án.

Bit `0` ở đầu (ví dụ `[0, 1, 1]`) không gây lỗi: `0·2 + 0 = 0`, giống như số `011₂ = 3`.

## 14. Độ phức tạp và so sánh

Gọi `n` = số nút (≤ 30).

| Cách | Thời gian | Bộ nhớ phụ | Ghi chú |
|---|---|---|---|
| 1. Chuỗi + `int(s, 2)` | `O(n)` | `O(n)` | ngắn, dựa vào thư viện |
| 2. List + lũy thừa | `O(n)` | `O(n)` | hiểu rõ trọng số |
| 3. Horner `num*2 + bit` | `O(n)` | `O(1)` | lời giải chính |
| 4. Bit `(num << 1) \| bit` | `O(n)` | `O(1)` | cùng ý Horner |
| 5. Đệ quy | `O(n)` | `O(n)` call stack | luyện đệ quy |

## 15. Bộ test

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.getDecimalValue(build_list([1, 0, 1])), 5)
check("đề 2", sol.getDecimalValue(build_list([0])), 0)
check("một bit 1", sol.getDecimalValue(build_list([1])), 1)
check("số 0 ở đầu", sol.getDecimalValue(build_list([0, 0, 1, 1])), 3)
check("toàn 0", sol.getDecimalValue(build_list([0] * 30)), 0)
check("30 bit 1", sol.getDecimalValue(build_list([1] * 30)), 2**30 - 1)
check("1 rồi 29 số 0", sol.getDecimalValue(build_list([1] + [0] * 29)), 2**29)

PASS đề 1: 5
PASS đề 2: 0
PASS một bit 1: 1
PASS số 0 ở đầu: 3
PASS toàn 0: 0
PASS 30 bit 1: 1073741823
PASS 1 rồi 29 số 0: 536870912


In [16]:
import random
random.seed(1290)
methods = [decimal_via_string, decimal_via_powers, decimal_via_reversed, decimal_horner, decimal_bitwise, decimal_recursive]
for _ in range(1000):
    bits = [random.randint(0, 1) for _ in range(random.randint(1, 30))]
    expected = int("".join(map(str, bits)), 2)
    for f in methods:
        assert f(build_list(bits)) == expected, (f.__name__, bits)
    assert Solution().getDecimalValue(build_list(bits)) == expected
print("1000 danh sách ngẫu nhiên: 6 cách + Solution đều khớp")

1000 danh sách ngẫu nhiên: 6 cách + Solution đều khớp


## 16. Lỗi người mới hay gặp

### Lỗi 1: Coi bit đầu là bit **thấp** nhất (đọc ngược trọng số)

Nhân trọng số tăng dần từ đầu danh sách (`2^0, 2^1, …`) là hiểu ngược đề: đề nói **MSB ở đầu**.

In [17]:
def wrong_lsb_first(head):
    total, weight = 0, 1
    node = head
    while node:
        total += node.val * weight          # SAI: bit đầu được trọng số 1
        weight *= 2
        node = node.next
    return total

print("sai :", wrong_lsb_first(build_list([1, 1, 0])), "| đúng:", Solution().getDecimalValue(build_list([1, 1, 0])))
print("ví dụ đề [1,0,1] đối xứng nên vẫn qua:", wrong_lsb_first(build_list([1, 0, 1])))

sai : 3 | đúng: 6
ví dụ đề [1,0,1] đối xứng nên vẫn qua: 5


### Lỗi 2: Cộng trước, nhân sau

`num = (num + val) * 2` nhân đôi **cả bit vừa đọc** → kết quả gấp đôi đáp án.

In [18]:
def wrong_add_then_mul(head):
    num = 0
    while head:
        num = (num + head.val) * 2          # SAI
        head = head.next
    return num

print("sai :", wrong_add_then_mul(build_list([1, 0, 1])), "| đúng:", Solution().getDecimalValue(build_list([1, 0, 1])))

sai : 10 | đúng: 5


### Lỗi 3: Quên `node = node.next` → vòng lặp vô hạn

Không cập nhật con trỏ thì `while node:` mãi đúng. Minh họa an toàn bằng cách giới hạn số vòng.

In [19]:
def wrong_forget_next(head, limit=5):
    num, node, steps = 0, head, 0
    while node and steps < limit:
        num = num * 2 + node.val
        steps += 1                           # quên node = node.next
    return num, steps

print(wrong_forget_next(build_list([1, 0, 1])), "<- bị kẹt ở nút đầu, phải cắt bởi limit")

(31, 5) <- bị kẹt ở nút đầu, phải cắt bởi limit


### Lỗi 4: Dùng `^` tưởng là lũy thừa

Trong Python, `2 ^ 3` là **XOR** (= 1), không phải `2³`. Lũy thừa là `2 ** 3` hoặc `1 << 3`.

In [20]:
print(2 ^ 3, 2 ** 3, 1 << 3)     # 1 8 8

1 8 8


### Lỗi 5: Dùng `+` thay `|` khi bit có thể > 1

Ở bài này `bit ∈ {0, 1}` nên `(num << 1) | bit` và `(num << 1) + bit` như nhau. Nhưng nếu "bit" thực ra là chữ số lớn hơn (biến thể cơ số khác), `|` sẽ **sai**. Hiểu rõ khi nào hai phép này trùng nhau.

In [21]:
num = 0b10
print((num << 1) | 1, (num << 1) + 1)     # 5 5 -> trùng khi bit là 0/1
print((num << 1) | 3, (num << 1) + 3)     # 7 7 -> vẫn trùng: 4 = 0b100 và 3 = 0b011 không có bit chung
print((5 << 1) | 3, (5 << 1) + 3)         # 11 13 -> khác! 10 = 0b1010 và 3 = 0b0011 chung bit 1

5 5
7 7
11 13


## 17. Mẹo nhận dạng pattern và biến thể

**Dấu hiệu:**
- "Đọc các chữ số từ trái (MSB) sang phải, không biết trước độ dài" → **Horner**: `num = num * base + digit`.
- "Duyệt linked list" → khuôn `node = head; while node: ...; node = node.next`.
- Nếu đọc từ phải (LSB) sang trái → cộng `digit * weight`, `weight *= base`.

**Bài liên quan:**
- LeetCode 2 *Add Two Numbers* (số lưu ngược trong linked list).
- LeetCode 445 *Add Two Numbers II* (số lưu xuôi — cần stack hoặc đảo danh sách).
- LeetCode 1017, 504 (đổi cơ số); LeetCode 8 *String to Integer (atoi)* (Horner với cơ số 10).
- LeetCode 1009 *Complement of Base 10 Integer* (thao tác bit).

In [22]:
# Mini: Horner tổng quát cho mọi cơ số, đọc từ trái sang phải
def horner(digits, base):
    num = 0
    for d in digits:
        num = num * base + d
    return num

print(horner([1, 0, 1], 2), horner([1, 2, 3], 10), horner([15, 15], 16), horner([2, 0, 2], 7))   # 5 123 255 100

5 123 255 100


## 18. Bài tập tự luyện

> Gợi ý ở mục 19, đáp án ở mục 20. Không gọi hàm TODO khi chạy notebook.

### Bài 1 — Tính tay
Giá trị của `[1, 0, 0, 1, 1]`? Gán vào `ex1_answer`.

In [23]:
ex1_answer = None   # TODO

# assert ex1_answer == Solution().getDecimalValue(build_list([1, 0, 0, 1, 1]))

### Bài 2 — Số thập phân → linked list nhị phân
Viết `int_to_bits_list(num)` trả về `head` của danh sách bit (MSB ở đầu). `0` → `[0]`.

In [24]:
def int_to_bits_list(num: int) -> "ListNode":
    # TODO
    pass

# assert to_list(int_to_bits_list(5)) == [1, 0, 1]
# assert to_list(int_to_bits_list(0)) == [0]

### Bài 3 — LSB ở đầu
Danh sách giờ lưu bit **thấp nhất ở đầu**. Viết `decimal_lsb_first(head)` trong `O(1)` bộ nhớ.

In [25]:
def decimal_lsb_first(head: "ListNode") -> int:
    # TODO
    pass

# assert decimal_lsb_first(build_list([1, 1, 0])) == 3      # 011 đọc ngược
# assert decimal_lsb_first(build_list([0, 0, 1])) == 4

### Bài 4 — Danh sách chữ số thập phân
Mỗi nút là một chữ số `0..9`, MSB ở đầu. Viết `decimal_digits_list(head)`.

In [26]:
def decimal_digits_list(head: "ListNode") -> int:
    # TODO
    pass

# assert decimal_digits_list(build_list([4, 0, 7])) == 407

### Bài 5 — Mini challenge: chia hết cho 3?
Cho danh sách bit (MSB ở đầu) **rất dài** (có thể hàng triệu nút). Trả `True` nếu số chia hết cho 3, **không** dựng số nguyên lớn — chỉ giữ phần dư.

In [27]:
def divisible_by_3(head: "ListNode") -> bool:
    # TODO: Horner theo modulo 3
    pass

# assert divisible_by_3(build_list([1, 1])) == True          # 3
# assert divisible_by_3(build_list([1, 0, 1])) == False      # 5
# assert divisible_by_3(build_list([1, 1, 0])) == True       # 6

## 19. Gợi ý 3 mức

**Bài 1:** (1) Horner. (2) 1 → 2 → 4 → 9 → ? (3) `9·2 + 1`.

**Bài 2:** (1) Lấy bit bằng `num % 2`, `num //= 2` (thu được từ LSB). (2) Đảo ngược, hoặc **chèn vào đầu** danh sách mỗi lần. (3) `head = ListNode(num % 2, head)`.

**Bài 3:** (1) Bit đầu có trọng số 1. (2) Giữ `weight`, nhân 2 mỗi bước. (3) `total += node.val * weight`.

**Bài 4:** (1) Horner với cơ số 10. (2) `num = num * 10 + node.val`. (3) Khuôn duyệt như cũ.

**Bài 5:** (1) `(a·2 + b) mod 3` chỉ phụ thuộc `a mod 3`. (2) Giữ `r = (r * 2 + bit) % 3`. (3) Trả `r == 0`.

## 20. Đáp án tham khảo

In [28]:
ans_ex1 = 19


def ans_int_to_bits_list(num):
    if num == 0:
        return ListNode(0)
    head = None
    while num > 0:
        head = ListNode(num % 2, head)      # chèn vào đầu -> tự đúng thứ tự MSB trước
        num //= 2
    return head


def ans_decimal_lsb_first(head):
    total, weight = 0, 1
    while head:
        total += head.val * weight
        weight <<= 1
        head = head.next
    return total


def ans_decimal_digits_list(head):
    num = 0
    while head:
        num = num * 10 + head.val
        head = head.next
    return num


def ans_divisible_by_3(head):
    r = 0
    while head:
        r = (r * 2 + head.val) % 3
        head = head.next
    return r == 0


assert ans_ex1 == Solution().getDecimalValue(build_list([1, 0, 0, 1, 1]))
assert to_list(ans_int_to_bits_list(5)) == [1, 0, 1]
assert to_list(ans_int_to_bits_list(0)) == [0]
assert ans_decimal_lsb_first(build_list([1, 1, 0])) == 3
assert ans_decimal_lsb_first(build_list([0, 0, 1])) == 4
assert ans_decimal_digits_list(build_list([4, 0, 7])) == 407
assert ans_divisible_by_3(build_list([1, 1])) == True
assert ans_divisible_by_3(build_list([1, 0, 1])) == False
assert ans_divisible_by_3(build_list([1, 1, 0])) == True
for v in range(0, 3000):
    head = ans_int_to_bits_list(v)
    assert Solution().getDecimalValue(head) == v                      # khứ hồi
    assert ans_divisible_by_3(head) == (v % 3 == 0)
print("Đáp án bài tập: tất cả assert PASS")

Đáp án bài tập: tất cả assert PASS


## 21. Template nhớ nhanh + checklist submit

```python
class Solution:
    def getDecimalValue(self, head) -> int:
        num = 0
        while head:
            num = num * 2 + head.val      # hoặc (num << 1) | head.val
            head = head.next
        return num
```

**Câu thần chú:** *"Đọc từ trái: nhân cơ số rồi cộng chữ số (Horner)."*

**Checklist:**
- [ ] Nhân **trước**, cộng **sau**: `num * 2 + val`.
- [ ] Bit đầu danh sách là bit **cao nhất**.
- [ ] Có `node = node.next` trong vòng lặp.
- [ ] Lũy thừa dùng `**` hoặc `<<`, **không** dùng `^`.
- [ ] Test số có bit `0` ở đầu và danh sách 1 nút `[0]`.